# Real-World Network Entropy Experiment

Same protocol as the synthetic experiment, applied to the **60 real-world networks** sampled across four domains:

| Domain | n |
|---|---|
| Biological | 15 |
| Economic | 15 |
| Social | 15 |
| Transportation | 15 |

**Measures computed:**

1. **Spectral (arithmetic compression) entropy** at four reduction levels: 80 %, 60 %, 40 %, 20 %  
2. **GCN link-prediction entropy + AUC** on the original (100 %) graph only

## 1. Imports & Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys, pickle, warnings
warnings.filterwarnings('ignore')

project_root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx

from pygsp import graphs as pygsp_graphs

from algorithm.coarsening_utils import coarsen, get_entropy_metadata_aritmethicEncoding
import algorithm.entropia_link_prediction as elp

print('All imports OK')

## 2. Load Real Networks

In [ ]:
PKL_PATH = os.path.join(
    os.getcwd(), 'LinkPrediction_Experiments', 'Final_sampled_networks.pkl'
)

with open(PKL_PATH, 'rb') as f:
    networks_df = pickle.load(f)

REDUCTION_LEVELS = [80, 60, 40, 20]
GCN_EPOCHS = 100

def build_graph(row):
    is_directed = 'Directed' in row['graphProperties']
    G = nx.DiGraph() if is_directed else nx.Graph()
    G.add_nodes_from(np.array(row['nodes_id']))
    G.add_edges_from(np.array(row['edges_id']))
    G = nx.to_undirected(G)   # coarsening requires undirected
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
        G = nx.convert_node_labels_to_integers(G)
    return G

print(f'Loaded {len(networks_df)} networks')
print('Domains:', networks_df['networkDomain'].value_counts().to_dict())
networks_df[['title', 'networkDomain', 'number_nodes', 'number_edges']].head(10)

## 3. Entropy Experiment

For each real network:
1. Spectral coarsening → **arithmetic compression entropy** at 80 %, 60 %, 40 %, 20 %
2. **GCN link-prediction entropy + AUC** on the original 100 % graph

In [ ]:
from tqdm.notebook import tqdm

results = []   # one row per (network × measure × level)

for _, row in tqdm(networks_df.iterrows(), total=len(networks_df), desc='Networks'):
    domain = row['networkDomain']
    name   = row.get('network_name', row.get('title', 'unknown'))

    G = build_graph(row)
    n = G.number_of_nodes()
    e = G.number_of_edges()

    # ── Spectral entropy at each reduction level ───────────────────────────
    W  = nx.to_scipy_sparse_array(G)
    Gp = pygsp_graphs.Graph(W)

    for r_pct in REDUCTION_LEVELS:
        spectral_norm = spectral_raw = None
        try:
            C, Gc, _, __ = coarsen(Gp, K=10, r=1 - r_pct / 100)
            G_red = nx.from_scipy_sparse_array(Gc.W)
            ent   = get_entropy_metadata_aritmethicEncoding(G_red)
            spectral_norm = ent['Entropy Normalizado']
            spectral_raw  = ent['Grafo']
        except Exception as exc:
            print(f'  Coarsen failed ({name}, {r_pct}%): {exc}')

        results.append({
            'domain':  domain,
            'name':    name,
            'n_nodes': n,
            'n_edges': e,
            'level':   r_pct,
            'measure': 'spectral_entropy',
            'value':   spectral_norm,
            'extra':   spectral_raw,
        })

    # ── GCN link prediction on original (100 %) graph ─────────────────────
    gcn_norm = gcn_auc = None
    try:
        gcn = elp.compare_real_vs_random_gcn(G, epochs=GCN_EPOCHS)
        if gcn and gcn['random_graph']['entropy'] not in (0, None):
            gcn_norm = gcn['real_graph']['entropy'] / gcn['random_graph']['entropy']
        gcn_auc = gcn['real_graph'].get('auc') if gcn else None
    except Exception as exc:
        print(f'  GCN failed ({name}): {exc}')

    results.append({
        'domain':  domain,
        'name':    name,
        'n_nodes': n,
        'n_edges': e,
        'level':   100,
        'measure': 'gcn_entropy',
        'value':   gcn_norm,
        'extra':   gcn_auc,   # AUC stored in 'extra' for GCN rows
    })

df = pd.DataFrame(results)
print(f'\nTotal rows: {len(df)}')
df.head(10)

## 4. Results Summary

In [ ]:
spectral_df = df[df['measure'] == 'spectral_entropy']
gcn_df      = df[df['measure'] == 'gcn_entropy']

domains = sorted(df['domain'].dropna().unique())

# Spectral entropy pivot
pivot = spectral_df.pivot_table(
    index='domain', columns='level', values='value', aggfunc='mean'
).round(4)
pivot.columns = [f'{c}%' for c in pivot.columns]
print('Mean normalised spectral entropy by domain & reduction level:')
display(pivot)

# GCN summary
gcn_summary = gcn_df.groupby('domain')[['value', 'extra']].agg(['mean', 'std']).round(4)
gcn_summary.columns = ['gcn_entropy_mean', 'gcn_entropy_std', 'auc_mean', 'auc_std']
print('\nGCN link-prediction entropy & AUC by domain (100 % graph):')
display(gcn_summary)

## 5. Visualisations

### 5a. Spectral entropy across reduction levels

In [ ]:
colors = cm.Set2(np.linspace(0, 1, len(domains)))
levels = sorted(REDUCTION_LEVELS, reverse=True)   # 80 → 20

fig, ax = plt.subplots(figsize=(9, 5))

for color, domain in zip(colors, domains):
    sub   = spectral_df[spectral_df['domain'] == domain]
    means = sub.groupby('level')['value'].mean().reindex(levels)
    stds  = sub.groupby('level')['value'].std().reindex(levels)

    ax.plot(levels, means, marker='o', linewidth=2, markersize=7,
            label=domain, color=color)
    ax.fill_between(levels, means - stds, means + stds, alpha=0.15, color=color)

ax.set_xlabel('Kept graph portion (%)', fontsize=12)
ax.set_ylabel('Normalised spectral entropy', fontsize=12)
ax.set_title('Spectral entropy vs. graph reduction level\nby network domain', fontsize=13)
ax.set_xticks(levels)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('real_spectral_entropy_reduction.png', dpi=150, bbox_inches='tight')
plt.show()

### 5b. GCN link-prediction entropy & AUC per domain

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, ylabel, title in zip(
    axes,
    ['value', 'extra'],
    ['Normalised GCN entropy', 'AUC'],
    ['GCN link-prediction entropy (normalised)', 'GCN link-prediction AUC']
):
    means = gcn_df.groupby('domain')[col].mean().reindex(domains)
    stds  = gcn_df.groupby('domain')[col].std().reindex(domains)

    ax.bar(domains, means, yerr=stds, color=colors, capsize=5,
           alpha=0.85, edgecolor='black', linewidth=0.6)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(len(domains)))
    ax.set_xticklabels(domains, rotation=15, ha='right', fontsize=9)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.suptitle('GCN link prediction on original (100 %) real networks', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('real_gcn_entropy_by_domain.png', dpi=150, bbox_inches='tight')
plt.show()

### 5c. Combined view: spectral entropy trajectory + GCN baseline

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for color, domain in zip(colors, domains):
    sub   = spectral_df[spectral_df['domain'] == domain]
    means = sub.groupby('level')['value'].mean().reindex(levels)
    ax.plot(levels, means, marker='o', linewidth=2, markersize=7,
            color=color, label=domain)

    gcn_mean = gcn_df[gcn_df['domain'] == domain]['value'].mean()
    if gcn_mean is not None and not np.isnan(float(gcn_mean)):
        ax.scatter([100], [gcn_mean], marker='D', s=80, color=color,
                   zorder=5, edgecolors='black', linewidths=0.6)
        ax.annotate('GCN', xy=(100, gcn_mean), xytext=(102, gcn_mean),
                    fontsize=7, color=color, va='center')

ax.axvline(x=100, color='grey', linewidth=0.8, linestyle=':')
ax.set_xlabel('Graph portion (%)', fontsize=12)
ax.set_ylabel('Normalised entropy', fontsize=12)
ax.set_title('Spectral entropy at reduction levels (lines)\n+ GCN link-prediction entropy at 100% (diamonds)', fontsize=12)
ax.set_xticks([20, 40, 60, 80, 100])
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('real_combined_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Results

In [ ]:
df.to_csv('real_networks_entropy_results.csv', index=False)
print('Results saved to real_networks_entropy_results.csv')
df.tail()